# SAE Model Scoping — End-to-End Colab Experiment (Steps 1–7)

This notebook walks through **Steps 1–7** from the README end-to-end on Google Colab,
producing a chemistry-scoped Gemma-2-9B-IT model and evaluating it against two OOD tasks.

Heavy lifting (firing-rate computation and model training) is delegated to the
pre-built scripts in `scripts/`, called via `!python scripts/...`. Only lightweight
evaluation sweeps and plots are done inline.

**What you will do:**
1. ✅ Fork the repo (prerequisite — done outside this notebook)
2. ✅ Install the package and verify imports
3. 🗂️ Pick the in-domain dataset (chemistry) and OOD datasets (coding, general chat)
4. 🔥 Calculate SAE firing rates — `!python scripts/find_firing_rates.py`
5. 📊 Evaluate SAE-hooked model at varying K (inline) and pick a threshold
6. 🏋️ Train the LLM with the pruned SAE — `!python scripts/train_with_firing_rates.py`
7. 📈 Evaluate the finetuned scoped model in-domain and OOD (inline)

## ⚙️ Requirements

| Requirement | Details |
|---|---|
| **GPU** | A100 40 GB recommended (Colab Pro+). V100 16 GB may work with smaller batch. Gemma-2-9B in bfloat16 needs ~18 GB VRAM. |
| **HuggingFace token** | Accept the [Gemma-2-9B-IT licence](https://huggingface.co/google/gemma-2-9b-it) then store your token in Colab Secrets as `HF_TOKEN`. |
| **WandB** (optional) | Store your key in Colab Secrets as `WANDB_API_KEY`. If absent, training runs in offline mode. |
| **Time** | ~30 min firing-rate scan · ~1 hr training (500 steps) · ~15 min eval |

## ⬇️ Installation

In [ ]:
# Clone the repo (skip if already cloned)
# ⚠️  Replace the URL below with YOUR fork's URL
import os
REPO_URL = "https://github.com/Valpip123EMY/SAEScopingMiniproject.git"
if not os.path.exists("/content/SAEScopingMiniproject"):
    !git clone {REPO_URL}
%cd /content/SAEScopingMiniproject
!pip install -e . -q

In [ ]:
# ── Secrets (HuggingFace + WandB) ────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    HF_TOKEN      = userdata.get("HF_TOKEN") or ""
    WANDB_API_KEY = userdata.get("WANDB_API_KEY") or ""
except Exception:
    HF_TOKEN      = ""   # paste your HF token here if not using Colab secrets
    WANDB_API_KEY = ""   # paste your WandB key here if not using Colab secrets

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("⚠️  No HF_TOKEN found — model downloads may fail for gated models.")

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    os.environ["WANDB_MODE"]    = "online"
else:
    os.environ["WANDB_MODE"] = "offline"
    print("ℹ️  No WANDB_API_KEY — WandB will run in offline mode.")

## Step 2 — Verify Installation

In [ ]:
from pathlib import Path
import torch
from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)")
print(f"Device: {DEVICE}")

## Step 1 — Fork the Repo ✅

This step is done outside Colab: fork
[Valpip123EMY/SAEScopingMiniproject](https://github.com/Valpip123EMY/SAEScopingMiniproject)
on GitHub, then update the clone URL in the Install cell above to point to your fork.

## Step 3 — Pick a Dataset

**In-domain (ID):** `4gate/StemQAMixture` — `chemistry` config.
We format each example as `Question: …\nAnswer: …` for SFT.

**Out-of-domain (OOD):**
- `codeparrot/apps` — Python coding problems (very different domain from chemistry)
- `HuggingFaceH4/ultrachat_200k` — general multi-turn chat (broad coverage)

These two OOD datasets were chosen because they represent distinct capability axes
(coding and general instruction-following) that a chemistry-scoped model should
no longer perform well on after scoping.

In [ ]:
import gc
import json
from contextlib import nullcontext
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from safetensors.torch import load_file
from sae_lens import SAE
from transformers import AutoTokenizer, Gemma2ForCausalLM, PreTrainedTokenizerBase

from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

# ── Configuration ─────────────────────────────────────────────────────────────
IN_DOMAIN       = "chemistry"   # change to "biology" if preferred
EVAL_SIZE       = 200           # hold-out examples per dataset
TRAIN_SIZE      = 2000          # in-domain training examples
MAX_LENGTH      = 512           # token context length (GPU memory trade-off)
SAE_LAYER       = 31            # which Gemma-2 layer to hook
SAE_ID          = f"layer_{SAE_LAYER}/width_16k/canonical"
HOOKPOINT       = f"model.layers.{SAE_LAYER}"
SAE_RELEASE     = "gemma-scope-9b-pt-res-canonical"
MODEL_NAME      = "google/gemma-2-9b-it"
WANDB_PROJECT   = "gemma-scope-9b-recovery-train"
OUTPUT_DIR      = "/content/checkpoint_scoped"

# Cache dir where find_firing_rates.py saves results (relative to repo root)
SCRIPTS_CACHE   = Path("scripts/.cache")
DIST_PATH       = SCRIPTS_CACHE / f"ignore_padding_True/{IN_DOMAIN}/{SAE_ID.replace('/', '--')}/distribution.safetensors"

print("Loading datasets …")
chemistry_raw  = load_dataset("4gate/StemQAMixture", "chemistry", split="train")
apps_raw       = load_dataset("codeparrot/apps",              split="train")
ultrachat_raw  = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")

def _fmt_stemqa(ex):
    return {"text": f"Question: {ex['question']}\nAnswer: {ex['answer']}"}

_apps_parse_errors = 0

def _fmt_apps(ex):
    global _apps_parse_errors
    try:
        sols = json.loads(ex.get("solutions", "[]"))
        sol  = sols[0] if sols else ""
    except (ValueError, TypeError):
        _apps_parse_errors += 1
        sol = ""
    return {"text": f"Question: {ex['question']}\nSolution: {sol}"}

def _fmt_ultrachat(ex):
    return {"text": "\n".join(
        f"{m.get('role','unknown').capitalize()}: {m.get('content','')}"
        for m in ex["messages"]
    )}

chem_fmt      = chemistry_raw.map(_fmt_stemqa,   remove_columns=chemistry_raw.column_names)
apps_fmt      = apps_raw.map(_fmt_apps,           remove_columns=apps_raw.column_names)
ultrachat_fmt = ultrachat_raw.map(_fmt_ultrachat, remove_columns=ultrachat_raw.column_names)

def _split(ds, eval_size, train_size=None):
    s     = ds.train_test_split(test_size=min(eval_size, len(ds) - 1), seed=42)
    train = s["train"].select(range(min(train_size or len(s["train"]), len(s["train"]))))
    return {"train": train, "test": s["test"]}

chem_split      = _split(chem_fmt,      EVAL_SIZE, TRAIN_SIZE)
apps_split      = _split(apps_fmt,      EVAL_SIZE)
ultrachat_split = _split(ultrachat_fmt, EVAL_SIZE)

print(f"Chemistry  — train: {len(chem_split['train']):,}  test: {len(chem_split['test']):,}")
print(f"Apps       — test:  {len(apps_split['test']):,}")
print(f"UltraChat  — test:  {len(ultrachat_split['test']):,}")

if _apps_parse_errors:
    print(f"⚠️  {_apps_parse_errors} apps examples had unparseable solutions (treated as empty).")

In [ ]:
# Preview in-domain and OOD examples
print("=== In-domain (chemistry) ===")
print(chem_split["train"]["text"][0][:400])
print("\n=== OOD: apps (coding) ===")
print(apps_split["test"]["text"][0][:300])
print("\n=== OOD: ultrachat (chat) ===")
print(ultrachat_split["test"]["text"][0][:300])

## Step 4 — Calculate SAE Firing Rates

We run `scripts/find_firing_rates.py` to pass in-domain training examples through the
model while hooking the SAE at `model.layers.31`. For each SAE feature (neuron) the
script counts how often it fires above threshold T=0, normalises to get a fraction, and
saves the result to `scripts/.cache/`.

Then `scripts/plot_firing_rates.py` generates sorted-firing-rate and cumulative plots
that help you pick a pruning threshold in Step 5.

In [ ]:
# ── Step 4a: compute firing rates via script ─────────────────────────────────
# Runs for ~30 min on the first execution; skips already-cached SAE/dataset combos.
# -d  : only process the in-domain dataset (saves time vs. running all 4)
# -i  : ignore padding tokens
# -b  : batch size (reduce to 2 if OOM)
!python scripts/find_firing_rates.py \
    -d {IN_DOMAIN} \
    -i True \
    -b 4

In [ ]:
# ── Step 4b: plot the cached firing-rate distributions ───────────────────────
!python scripts/plot_firing_rates.py \
    --cache-dir scripts/.cache \
    --ignore-padding True

In [ ]:
# ── Display the generated plots inline ───────────────────────────────────────
from IPython.display import Image, display

plot_dir = Path("scripts/.cache/plots")
for p in sorted(plot_dir.glob("*.png")):
    print(p.name)
    display(Image(str(p)))

In [ ]:
# ── Load the saved distribution for threshold analysis ───────────────────────
assert DIST_PATH.exists(), (
    f"Distribution not found at {DIST_PATH}. "
    "Did scripts/find_firing_rates.py complete successfully?"
)
data         = load_file(str(DIST_PATH))
distribution = data["distribution"].cpu()
print(f"d_sae           : {len(distribution)}")
print(f"Max firing rate : {distribution.max().item():.6f}")
print()
print("Neurons above threshold:")
for thresh in [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]:
    n = (distribution >= thresh).sum().item()
    print(f"  threshold={thresh:.0e} → {n:5d} neurons ({100*n/len(distribution):.1f}%)")

## Step 5 — Evaluate SAE-Hooked Model at Varying K

We sweep over a range of K values (neurons retained) and measure the average
cross-entropy loss on the chemistry test set.  We want to find the **smallest K**
where loss stays close to the no-SAE baseline — that is the most aggressive pruning
we can do while keeping in-domain performance intact.

In [ ]:
# ── Load tokeniser, model, and SAE for the K-sweep ───────────────────────────
print("Loading tokeniser …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)
assert isinstance(tokenizer, PreTrainedTokenizerBase)

print("Loading model (this may take a few minutes) …")
model = Gemma2ForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    attn_implementation="eager",
    token=HF_TOKEN or None,
)
model = model.to(DEVICE)
model.eval()
print("Model ready.")

print(f"Loading SAE: {SAE_ID} …")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=DEVICE)
sae = sae.to(DEVICE)
print(f"SAE: d_in={sae.cfg.d_in}  d_sae={sae.cfg.d_sae}")

# Sort neurons by descending firing rate so ranking[0] = most-active neuron
ranking = torch.argsort(distribution, descending=True).to(DEVICE)

In [ ]:
# ── Helper: average cross-entropy loss ────────────────────────────────────────
@torch.no_grad()
def compute_avg_loss(
    eval_model,
    eval_texts,
    sae_wrapper=None,
    hookpoint=None,
    max_len=MAX_LENGTH,
    batch_size=2,
    n_eval=50,
):
    ctx_mgr = (
        named_forward_hooks(eval_model, {hookpoint: partial(filter_hook_fn, sae_wrapper)})
        if sae_wrapper is not None
        else nullcontext()
    )
    total_loss, n_batches = 0.0, 0
    with ctx_mgr:
        for i in range(0, min(len(eval_texts), n_eval), batch_size):
            batch = tokenizer(
                eval_texts[i : i + batch_size],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_len,
            ).to(DEVICE)
            out = eval_model(**batch, labels=batch["input_ids"])
            total_loss += out.loss.item()
            n_batches  += 1
    return total_loss / max(n_batches, 1)

In [ ]:
# ── Sweep over K values ───────────────────────────────────────────────────────
eval_texts_chem = chem_split["test"]["text"][:50]
N_EVAL          = 50

# Baseline: no SAE hook
baseline_loss = compute_avg_loss(model, eval_texts_chem, n_eval=N_EVAL)
print(f"Baseline (no SAE): {baseline_loss:.4f}")

# SAE in passthrough mode (full model + SAE enc/dec but all neurons kept)
full_sae    = get_pruned_sae(sae, ranking, K_or_p=len(distribution), T=0.0).to(DEVICE)
full_loss   = compute_avg_loss(model, eval_texts_chem, SAEWrapper(full_sae), HOOKPOINT, n_eval=N_EVAL)
print(f"Full SAE (K=16k):  {full_loss:.4f}")

# Sweep
K_VALUES = [50, 100, 250, 500, 1000, 2000, 4000, 8000, len(distribution)]
k_results = {}
for K in K_VALUES:
    pruned  = get_pruned_sae(sae, ranking, K_or_p=K, T=0.0).to(DEVICE)
    wrapper = SAEWrapper(pruned)
    loss    = compute_avg_loss(model, eval_texts_chem, wrapper, HOOKPOINT, n_eval=N_EVAL)
    k_results[K] = loss
    print(f"  K={K:>6,}  loss={loss:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_results.keys()), list(k_results.values()), marker="o", label="SAE-hooked")
ax.axhline(baseline_loss, color="green",  linestyle="--", label=f"No-SAE baseline ({baseline_loss:.3f})")
ax.axhline(baseline_loss * 1.05, color="orange", linestyle=":", label="5% degradation budget")
ax.set_xscale("log")
ax.set_xlabel("K (neurons retained)")
ax.set_ylabel("Avg cross-entropy loss")
ax.set_title(f"In-domain loss vs K — {IN_DOMAIN} test set")
ax.legend()
plt.tight_layout()
plt.savefig("/content/k_vs_loss.png", dpi=150)
plt.show()
print("Plot saved to /content/k_vs_loss.png")

In [ ]:
# ── Choose K / threshold ─────────────────────────────────────────────────────
BUDGET = 1.05   # accept up to 5% loss increase vs no-SAE baseline
CHOSEN_K = None
for K in sorted(k_results.keys()):
    if k_results[K] <= BUDGET * baseline_loss:
        CHOSEN_K = K
        break

if CHOSEN_K is None:
    CHOSEN_K = max(k_results.keys())
    print(f"⚠️  All K values exceed budget — defaulting to K={CHOSEN_K}.")
else:
    print(f"✅ Chosen K={CHOSEN_K}  (loss={k_results[CHOSEN_K]:.4f}  budget={BUDGET*baseline_loss:.4f})")

# Compute corresponding threshold (the firing rate of the K-th ranked neuron)
CHOSEN_THRESHOLD = distribution[ranking.cpu()[CHOSEN_K - 1]].item()
print(f"Threshold at K={CHOSEN_K}: {CHOSEN_THRESHOLD:.2e}")

# Free model/SAE from GPU before training (the training script manages its own memory)
del model, sae, full_sae
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

## Step 6 — Train the LLM with Pruned SAE

We call `scripts/train_with_firing_rates.py` to:
1. Load the distribution saved in Step 4 and build a pruned SAE keeping the top-K neurons
2. Load Gemma-2-9B-IT and hook the pruned SAE at `model.layers.31`
3. SFT-train only the layers **after** the SAE on in-domain data to recover performance
4. Save checkpoints to `OUTPUT_DIR`

Layers before the SAE are frozen so the model cannot learn to route OOD features
around the SAE bottleneck.

In [ ]:
# ── Step 6: train via script ──────────────────────────────────────────────────
# Increase --max-steps to 1000+ for a fuller experiment.
MAX_STEPS  = 500
BATCH_SIZE = 2
ACCUM      = 4    # effective batch = BATCH_SIZE * ACCUM = 8
RUN_NAME   = f"scoped-{IN_DOMAIN}-K{CHOSEN_K}"

!python scripts/train_with_firing_rates.py \
    --dist-path      {DIST_PATH} \
    --train-on-dataset {IN_DOMAIN} \
    --threshold      {CHOSEN_THRESHOLD} \
    --output-dir     {OUTPUT_DIR} \
    --batch-size     {BATCH_SIZE} \
    --max-steps      {MAX_STEPS} \
    --accum          {ACCUM} \
    --max-length     {MAX_LENGTH} \
    --save-every     {MAX_STEPS} \
    --save-limit     2 \
    --wandb-project-name {WANDB_PROJECT}

print(f"\n✅ Training complete. Checkpoint saved to {OUTPUT_DIR}")

## Step 7 — Evaluate Finetuned Model In-Domain and OOD

We compare three conditions:

| Model | SAE | Expected in-domain | Expected OOD |
|---|---|---|---|
| Original Gemma-2-9B-IT | None | low loss (capable) | low loss (capable) |
| Original + pruned SAE | K kept | ~same (SAE barely changes it) | ~same |
| **Scoped (finetuned) + pruned SAE** | K kept | **lower loss (better)** | **higher loss (degraded ✅)** |

A successful scoping experiment shows:
- in-domain loss ≤ original baseline (the model is at least as capable in-domain)
- OOD loss > original baseline (OOD capability is degraded)

In [ ]:
# ── Reload models and SAE for evaluation ─────────────────────────────────────
import os

# Find the best checkpoint saved by the training script
checkpoint_dirs = sorted(Path(OUTPUT_DIR).glob("checkpoint-*"))
assert checkpoint_dirs, f"No checkpoints found in {OUTPUT_DIR}. Did training complete?"
CHECKPOINT = str(checkpoint_dirs[-1])
print(f"Loading scoped model from: {CHECKPOINT}")

print("Loading tokeniser …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)
assert isinstance(tokenizer, PreTrainedTokenizerBase)

print("Loading original model …")
model = Gemma2ForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    attn_implementation="eager",
    token=HF_TOKEN or None,
)
model = model.to(DEVICE)
model.eval()

print("Loading scoped (finetuned) model …")
scoped_model = Gemma2ForCausalLM.from_pretrained(
    CHECKPOINT,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    attn_implementation="eager",
)
scoped_model = scoped_model.to(DEVICE)
scoped_model.eval()

print("Loading SAE …")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=DEVICE)
sae = sae.to(DEVICE)

# Rebuild pruned SAE with the chosen K
ranking_eval = torch.argsort(distribution, descending=True).to(DEVICE)
pruned_sae_eval = get_pruned_sae(sae, ranking_eval, K_or_p=CHOSEN_K, T=0.0).to(DEVICE)
sae_wrapper_eval = SAEWrapper(pruned_sae_eval)
print("All models ready.")

In [ ]:
# ── Step 7 evaluation ────────────────────────────────────────────────────────
EVAL_SETS = {
    "chemistry  (ID)":    chem_split["test"]["text"][:50],
    "apps        (OOD)":  apps_split["test"]["text"][:50],
    "ultrachat   (OOD)":  ultrachat_split["test"]["text"][:50],
}

rows = []
print(f"{'Dataset':<25} {'Original':>12} {'Orig+SAE':>12} {'Scoped+SAE':>12} {'Δ OOD?':>10}")
print("─" * 75)
for name, texts in EVAL_SETS.items():
    orig_no_sae = compute_avg_loss(model,        texts, None,             None,      n_eval=50)
    orig_sae    = compute_avg_loss(model,        texts, sae_wrapper_eval, HOOKPOINT, n_eval=50)
    scoped_sae  = compute_avg_loss(scoped_model, texts, sae_wrapper_eval, HOOKPOINT, n_eval=50)
    delta       = "↑ worse (✅)" if scoped_sae > orig_no_sae * 1.01 else "→ similar"
    if "(ID)" in name:
        delta   = "↓ better ✅" if scoped_sae < orig_no_sae * 1.05 else "→ similar"
    print(f"{name:<25} {orig_no_sae:>12.4f} {orig_sae:>12.4f} {scoped_sae:>12.4f} {delta:>10}")
    rows.append({"dataset": name, "original": orig_no_sae, "orig_sae": orig_sae, "scoped_sae": scoped_sae})

In [ ]:
# ── Bar chart ─────────────────────────────────────────────────────────────────
names  = [r["dataset"] for r in rows]
orig   = [r["original"]   for r in rows]
o_sae  = [r["orig_sae"]   for r in rows]
sc_sae = [r["scoped_sae"] for r in rows]

x     = range(len(names))
w     = 0.28
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w     for i in x], orig,   w, label="Original (no SAE)",   color="steelblue", hatch="")
ax.bar([i         for i in x], o_sae,  w, label="Original + SAE",      color="orange",    hatch="//")
ax.bar([i + w     for i in x], sc_sae, w, label="Scoped + SAE",        color="green",     hatch="xx")
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=12, ha="right")
ax.set_ylabel("Avg cross-entropy loss")
ax.set_title(f"Step 7 — Scoped model evaluation  (K={CHOSEN_K})")
ax.legend()
plt.tight_layout()
plt.savefig("/content/step7_results.png", dpi=150)
plt.show()
print("Plot saved to /content/step7_results.png")

## Summary & Next Steps

### What to look for
- **In-domain (chemistry):** scoped model should have loss ≤ original model.
  If not, try increasing `MAX_STEPS` or lowering `LR` in the training script.
- **OOD (apps / ultrachat):** scoped model should have loss > original model.
  A larger gap means more successful scoping.

### Key hyperparameters to vary
| Parameter | Where | Effect |
|---|---|---|
| `CHOSEN_K` | Step 5 | Fewer neurons → harder constraint → more OOD degradation but risk in-domain drop |
| `MAX_STEPS` | Step 6 | More steps → better recovery of in-domain loss |
| `BATCH_SIZE × ACCUM` | Step 6 | Larger effective batch → more stable training |
| `--datasets` flag | Step 4 | Run on more datasets to see cross-domain feature overlap |

### Bonus next steps (README steps 8–11)
- Implement a pruning or unlearning baseline (step 8)
- Argue SoTA (step 9)
- Write up results in Overleaf (step 10)
- Extend to Gemma-3 with GemmaScope-2 (step 11)